# [2.1: TF-IDF] // Content-based Filtering with TF-IDF Vectorization

In the previous module, we saw one of the main limitations of **collaborative filtering**: the **long-tail problem**. When users or items have very few interactions, there simply is not enough information to reliably compute similarities. As a result, collaborative filtering struggles with *new users*, *new items*, and generally sparse data.

This is where **content-based filtering** becomes useful. Instead of relying on user–item interactions, content-based methods compare items based on their **intrinsic properties**. If we can describe items in terms of their content, we can compute similarities even when interaction data is scarce—or entirely absent.

In this module, we focus on **textual content**.

We will work with **newspaper articles**. For news data, the long-tail problem is particularly severe: Articles have a very **short lifespan**. By the time enough users have interacted with an article to apply collaborative filtering, it is often already outdated.So, many articles receive only a handful of interactions, if any at all.
|
However, newspaper articles have one major advantage: **they contain a lot of content information**. The full text of an article provides a detailed description of what it is about. If we can represent that text in a structured way, we can compare articles directly (without relying on user behavior).

## NLP

Of course, we cannot directly compare two pieces of text using cosine similarity. Raw text is **unstructured data**: it consists of characters, punctuation, capitalization, HTML tags, line breaks, and other noise.

To make text usable for similarity computations, we first need to **structure** it. Turning raw text into structured representations is the domain of **Natural Language Processing (NLP)**. NLP is a large and complex field, but for our purposes we only need a small subset of its techniques.

Fortunately, we can rely on existing tools. We will use **spaCy**, a Python library that makes NLP accessible and practical. spaCy is designed for researchers and data scientists: it implements many standard NLP techniques and makes sensible algorithmic choices for you.

### Tokenization

In this submodule, we will mainly use spaCy for **tokenization** (the process of splitting text into individual words). Tokenization is a fundamental step in almost all NLP pipelines.

The idea of tokenization is that it removes irrelevant clutter such as punctuation and capitalization and gives us a clean list of words we can count, compare, and analyze.

You could, in principle, perform tokenization using regular expressions. However, natural language contains many edge cases, and spaCy handles these robustly and consistently.

### Lemmatization
Besides tokenization, spaCy also performs **lemmatization**, and it does so particularly well.

**Lemmatization** is the process of reducing words to their *base form* (the **lemma**). The goal is to treat different grammatical forms of the same word as a single concept. For example:

* *run*, *runs*, *running*, *ran* -> **run**
* *children* -> **child**
* *better* -> **good**
* *was*, *were* -> **be**

Without lemmatization, these variations would be treated as separate words, even though they clearly refer to the same underlying concept. 

## Installation

spaCy itself is already included in the Conda environment you installed at the start of the course. However, spaCy relies on **language models**, which must be downloaded separately.

To install the English language models, run the following commands in your terminal:

    python -m spacy download en_core_web_sm
    python -m spacy download en_core_web_md

Once these models are installed, you are ready to start processing text and building content-based representations of articles.

## Background

Many forms of content-based filtering ultimately aim to compute **similarities between items**. As we saw earlier with collaborative filtering (and with content-based filtering using genres), once we know how similar items are, we can make recommendations based on those similarities.

In the previous module, we did this using a **utility matrix** filled with user ratings. In that matrix, each user (in user-based CF) or each item (in item-based CF) was represented by a sequence of numbers—the ratings.

With **content-based filtering**, we want to follow the same general idea. The key difference is that instead of numbers representing *ratings*, the numbers will represent *content information*.

Associating an item with a sequence of numbers is called **vectorization**. Such a sequence of numbers is also referred to as a **vector**.

Even when we want to compare newspaper articles based on their text, our goal is ultimately the same: convert each article into a numerical vector so that we can compute similarities between articles.

There are multiple ways to perform this conversion. In this notebook, we will use **TF–IDF vectorization**. In the next notebook, you will explore a different approach.

### TF–IDF

TF–IDF stands for **term frequency–inverse document frequency**. The core idea is to assign a numerical score to each word in a document. This score reflects how **representative** that word is for the document.

Intuitively:

* Words that appear **often in a document** receive a higher score.
* Words that appear **in many documents** receive a lower score.

As a result, a high TF–IDF score indicates that a word is frequent in a specific text but rare across the collection as a whole (making it informative for distinguishing that text from others).

By computing such scores for all words, each document can be represented as a vector of numbers. These vectors can then be compared using similarity measures such as cosine similarity, enabling content-based recommendations.

In the rest of this notebook, you will learn how TF–IDF scores are computed and how they can be used in practice.

# Getting started

Start by loading the necessary libraries.

In [ ]:
import pandas as pd
import numpy as np
import spacy
import pooch

# model should already be downloaded, but if it doesn't work (copy this code in another cell and run)
# from spacy.cli import download 
# download("en_core_web_sm") 

%reload_ext autoreload
%autoreload 2

## Loading Data

We begin by loading subtitles from **ten randomly selected movies** to experiment with text-based similarity methods.

The data we use here is derived from the **MIND dataset** (*Microsoft News Dataset*). MIND was created by Microsoft Research to support research on **news recommendation systems**. It contains large-scale user interaction data together with rich textual content, such as article titles, abstracts, and metadata. The dataset was designed to enable the development and evaluation of both collaborative and content-based recommender systems, particularly in settings where textual information plays a central role.

Although MIND is primarily a **news** dataset, its structure (collections of documents with associated text) makes it well suited for experimenting with text-based similarity techniques more generally. In this notebook, we repurpose a small subset of textual data (movie subtitles) to illustrate these ideas in a more intuitive way.

We will *not* use the full dataset. Instead, we work with a **very small, carefully selected sample**. The main reason for this is practical: natural language processing operations are computationally expensive. Running the full pipeline (tokenization, lemmatization, vectorization) on the complete dataset would take many hours (ask us how we know 🙂). More importantly, you would not gain much additional insight from simply waiting for your code to finish.

By working with a small subset, we keep the notebook manageable and focused on understanding the underlying concepts rather than on computational scale.

Another reason for using a **curated subset** of the data is that the extreme *sparseness* of the full dataset (many users interacting with very few items) makes evaluation much more complex. Handling this properly requires a lot additional work and knowledge. For now, we want to focus on the core ideas of text representation and content-based similarity, without getting distracted by these complications.

So we will use this small dataset, but keep in mind that using the full set would make things substantially more complicated.

In [ ]:
# Download data
DATA_REPO = "https://raw.githubusercontent.com/uvapl/recommender-systems/2025/data/m2/"
for fname in ["MIND-micro-news.tsv", "MIND-micro-behaviors.tsv"]:
    pooch.retrieve(url = DATA_REPO + fname, known_hash=None, fname=fname, path="data", progressbar=True)
for fname in ["tests_m2.py"]:
    pooch.retrieve(url = DATA_REPO + fname, known_hash=None, fname=fname, path=".", progressbar=True)

import tests_m2

# Read data
# Column names (see MIND documentation)
columns_articles = [
    "NewsID",          # unique ID of the news article
    "Category",        # coarse-grained topic (e.g., Sports)
    "SubCategory",     # fine-grained topic (e.g., Football)
    "Title",           # news headline
    "Abstract",        # short summary
]
news_df = pd.read_csv("data/MIND-micro-news.tsv", sep="\t", index_col = "NewsID", header=None, names=columns_articles, encoding='utf-8')[["Title", "Abstract"]]

# Column names (see MIND documentation)
columns_impressions = [
    "ImpressionID",  # Unique identifier for session.
    "UserID",        # Anonymized unique user identifier (e.g., U0001). Each user can have multiple sessions.
    "Time",          # Timestamp of the impression session.
    "History",       # Space-separated list of NewsIDs the user clicked *before this session*.
    "Impressions",   # Space-separated list of candidate articles shown in this session, 
                     #   formatted as "NewsID-1" (clicked) or "NewsID-0" (ignored/not clicked).
]
impressions = pd.read_csv("data/MIND-micro-behaviors.tsv", sep="\t", index_col = "ImpressionID", header=None, names=columns_impressions, encoding='utf-8')
pd.set_option("display.max_colwidth", 100)
display(news_df)
display(impressions.fillna("")[["UserID", "Impressions"]].head(20))

### Question 1
*1 pt.*

The news data contains two text columns: **`Title`** and **`Abstract`**. Since we want to work with a single block of text per article, combine these columns into a new column called **`text`**. (Make sure to put space between the two fields.)

Implement here below.

In [ ]:
# your code here


In [ ]:
# test your solution

tests_m2.test_01(news_df)

# Processing the data

The first step is to **tokenize and lemmatize** the texts. To do this, you will use a few key concepts from **spaCy**:

* First, load the language model:

  ```
  nlp = spacy.load("en_core_web_sm")
  ```

* You can then use this model as a function that converts a string of text into a spaCy **Doc** object containing all detected tokens:

  ```
  doc = nlp(text)
  ```

* You can access individual tokens by iterating over the `Doc` object:

  ```
  for token in doc:
      ...
  ```

Each token has several useful properties:

* `token.lemma`
  Returns the **lemma** (base form) of the token.

* `token.is_alpha`
  A boolean indicating whether the token consists only of alphabetical characters (i.e., no numbers or punctuation).

* `token.is_stop`
  A boolean indicating whether the token is a **stop word**. These are very common words that contributes little meaning to the text (such as *“a”*, *“the”*, *“if”*, etc.).

Using these properties, you can filter out uninformative tokens and build a clean, lemmatized representation of each text.

### Question 2

*3 pts.*

Implement the function `tokenize_and_lemmatize()` below. The function should take a Pandas `Series` containing unprocessed text as input and return a `Series` of lists, where each list contains the **tokenized and lemmatized** representation of the corresponding text.

Make sure that:

* the output lists contain **lemmas**, not the original tokens,
* all lemmas are converted to **lowercase**,
* only tokens consisting of **alphabetic characters** are included,
* **stopwords** are excluded from the output.

In [ ]:
def tokenize_and_lemmatize(texts: pd.Series) -> pd.Series:
    # your code here
    
lemmas = tokenize_and_lemmatize(news_df["text"])
display(lemmas.head())

In [ ]:
# test your solution

tests_m2.test_02(tokenize_and_lemmatize)

# Computing TF–IDF

Now we are ready to compute **TF–IDF scores**. Our goal is to produce a matrix (a Pandas DataFrame) that contains a score for **every term in every document**:

<table id="T_8c0b3"><thead><tr><th class="blank level0" >&nbsp;</th><th id="T_8c0b3_level0_col0" class="col_heading level0 col0" >diplomat</th><th id="T_8c0b3_level0_col1" class="col_heading level0 col1" >treaty</th><th id="T_8c0b3_level0_col2" class="col_heading level0 col2" >sanction</th><th id="T_8c0b3_level0_col3" class="col_heading level0 col3" >summit</th><th id="T_8c0b3_level0_col4" class="col_heading level0 col4" >border</th><th id="T_8c0b3_level0_col5" class="col_heading level0 col5" >security</th><th id="T_8c0b3_level0_col6" class="col_heading level0 col6" >...</th></tr><tr><th class="index_name level0" >NewsID</th><th class="blank col0" >&nbsp;</th><th class="blank col1" >&nbsp;</th><th class="blank col2" >&nbsp;</th><th class="blank col3" >&nbsp;</th><th class="blank col4" >&nbsp;</th><th class="blank col5" >&nbsp;</th></tr></thead><tbody><tr><th id="T_8c0b3_level0_row0" class="row_heading level0 row0" >N0001</th><td id="T_8c0b3_row0_col0" class="data row0 col0" >0.24</td><td id="T_8c0b3_row0_col1" class="data row0 col1" >0.35</td><td id="T_8c0b3_row0_col2" class="data row0 col2" >0.17</td><td id="T_8c0b3_row0_col3" class="data row0 col3" >0.40</td><td id="T_8c0b3_row0_col4" class="data row0 col4" >0.20</td><td id="T_8c0b3_row0_col5" class="data row0 col5" >0.20</td><td>...</td></tr><tr><th id="T_8c0b3_level0_row1" class="row_heading level0 row1" >N0002</th><td id="T_8c0b3_row1_col0" class="data row1 col0" >0.00</td><td id="T_8c0b3_row1_col1" class="data row1 col1" >0.00</td><td id="T_8c0b3_row1_col2" class="data row1 col2" >0.55</td><td id="T_8c0b3_row1_col3" class="data row1 col3" >0.32</td><td id="T_8c0b3_row1_col4" class="data row1 col4" >0.00</td><td id="T_8c0b3_row1_col5" class="data row1 col5" >0.00</td><td>...</td></tr><tr><th id="T_8c0b3_level0_row2" class="row_heading level0 row2" >...</th><td>...</td><td>...</td><td>...</td><td>...</td><td>...</td><td>...</td><td>...</td></tr></tbody></table>

As discussed above, the score should reflect how **representative** a word is for a particular document. TF–IDF (*term frequency-inverse document frequency*) combines two intuitions:

* Words that appear **often in a document** receive a higher score (term frequency).
* Words that appear **in many documents** receive a lower score (inverse document frequency).

The exact definitions are given below.

## Term Frequency (TF)

When we want to understand how representative a term is for a particular document, the raw number of occurrences is not enough. Longer documents naturally contain more words, so they will also contain more occurrences of many terms.

Instead, we measure how often a term appears **relative to the total number of terms in that document**. This is the **term frequency**.

Example: if the term “diplomat” occurs 5 times in document $d$, and $d$ contains a total of 100 words, then:

$$
\text{tf}(t,d) = \frac{5}{100} = 0.05
$$

The formal definition of term frequency for term $t$ in document $d$ is:

$$
\text{tf}(t,d) = \frac{\text{number of occurances of t in d}}{\text{total number of terms in d}} = \frac{f_{t,d}}{\sum_{t' \in d} f_{t',d}}
$$

where $f_{t,d}$ is the number of times term $t$ appears in document $d$.

## Inverse Document Frequency (IDF)

TF alone still does not tell us which words are *informative*. Some words occur frequently in most texts (e.g., “come” or “go”). They may get a high TF score, but they are not distinctive.

To correct for this, we use **inverse document frequency (IDF)**. IDF is low for words that appear in many documents, and higher for words that appear in relatively few documents. It is a measure of how **distinctive** a term is within a collection of documents.

Suppose our dataset $D$ contains 20 documents, and the term “diplomat” appears in 4 of them. Then:

$$
\text{idf}(t,D) = \log\left(\frac{20}{4}\right) \approx 0.70
$$

The formal definition is:

$$
\text{idf}(t,D) = \log\left(\frac{\text{total number of documents}}{\text{number of documents where t occurs}}\right) = \log\left(\frac{|D|}{|D_t|}\right)
$$

where: $|D|$ is the total number of documents, and $|D_t|$ is the number of documents that contain term $t$.

Note that this formula uses a logarithm. There are several good reasons for using a log here, but they go beyond the scope of this assignment. If you'd like to learn more about it, you can delve into this article:

[Robertson, Stephen. "Understanding inverse document frequency: on theoretical arguments for IDF." *Journal of documentation* 60.5 (2004): 503-520.](https://dmice.ohsu.edu/bedricks/courses/cs635_spring_2017/pdf/robertson_2004.pdf)

## TF–IDF

Finally, we combine term frequency and inverse document frequency to obtain a score for how relevant a term is to a document:

$$
\text{tf-idf}(t,d,D) = \text{tf}(t,d) \cdot \text{idf}(t,D)
$$

Continuing the example above:

* $\text{tf}(t,d) \approx 0.05$
* $\text{idf}(t,D) \approx 0.70$

So:

$$
\text{tf-idf}(t,d,D) \approx 0.05 \cdot 0.70 \approx 0.035
$$


## Selecting vocabulary
When using **TF–IDF**, it is common to restrict the **document frequency (DF)** range of terms. In practice, this means keeping only words that are **neither too common nor too rare**.

The main motivation is **efficiency**. Each unique term becomes a dimension in the TF–IDF vectors. If we keep all terms, the vocabulary grows very large, leading to higher memory usage and slower similarity computations. By filtering terms based on DF, we keep the representation compact and computationally manageable.

There is also a conceptual benefit:

* **Very common terms** (high DF), such as words like *“go”* or *“them”*, appear in almost all documents. They do not help distinguish one document from another and therefore contribute little useful information—even with IDF weighting.
* **Very rare terms** (low DF), often typos or extremely specific names, tend to add noise and rarely improve generalization.

By restricting DF, we focus on terms that are informative **and** keep the vector space small enough to work with efficiently.


### Question 3

*3 pts.*

Complete the function `compute_term_document_counts()` below.
The function takes a Series of lemmatized documents as input and computes, for each term, **in how many documents it occurs** (document frequency). It should then select only those terms whose document frequency is **greater than or equal to `min_df`**, and **less than or equal to `max_df`**.

It should return a Pandas Series containing the document counts for the selected terms.

In [ ]:
def compute_term_document_counts(lemmas: pd.Series, min_df: float = 0.15, max_df: float = 0.5) -> pd.Series:
    # your code here

document_counts = compute_term_document_counts(lemmas)

In [ ]:
# test your solution

tests_m2.test_03(compute_term_document_counts)

### Question 4

*3 pts.*

Complete the function `compute_tf()` below. It should compute the **term frequency (TF)** score (as defined above) for each lemma in each document.

Your function should return a **term–document matrix** (a Pandas DataFrame) where:

* the **rows** represent documents,
* the **columns** represent terms (lemmas),
* each value is the **TF score** of that term in that document (i.e., how frequent the term is relative to the total number of terms in the document).

For efficiency, restrict the matrix to the terms in `vocab` only.

In [ ]:
def compute_tf(lemmas: pd.Series, vocab: list):
    # your code here

vocab = list(document_counts.index)
tf_scores = compute_tf(lemmas, vocab)

In [ ]:
# test your solution

tests_m2.test_04(compute_tf)

### Question 5

*2 pts.*

Complete the function `tf_idf()` below. It should take the `tf_scores` (the TF matrix from the previous question), the `document_counts` (the document frequency counts from before), and return a Pandas DataFrame containing the **TF–IDF score** for each term in each document.

The output should have the same shape as `tf_scores`: rows are documents, columns are terms, and each value is the TF–IDF score for that (document, term) pair.

In [ ]:
def compute_tfidf(tf, doc_counts):
    # your code here

tfidf_vectorization = compute_tfidf(tf_scores, document_counts)
display(tfidf_vectorization.style.format(precision=2).background_gradient())

In [ ]:
# test your solution
tests_m2.test_05(compute_tfidf)

# Similarity Matrix

The TF–IDF matrix constructed above is our **vectorization**: it represents each document as a vector of numerical values. As before, we can use these vectors to compute similarities between documents.

The procedure is the same as in earlier modules:

1. **Mean-center** the data.
2. Compute the **cosine similarity** between each pair of document vectors.

This results in a **similarity matrix**, which allows us to look up, for each document, how similar it is to every other document.

Since you have already implemented this process multiple times and the underlying logic has not changed, we provide the code for computing the cosine similarity matrix below.

In [ ]:
def mean_center(df: pd.DataFrame) -> pd.DataFrame:
    # subtract means from columns 
    return (df.T - df.mean(axis = 1)).T

def cosine_similarity_matrix(X: pd.DataFrame) -> pd.DataFrame:
    # normalize vectors
    X_norm = X.div(np.linalg.norm(X, axis=1), axis=0)

    # cosine similarity => dot product of normalized vectors
    sim = X_norm @ X_norm.T
    return pd.DataFrame(sim, index = X.index, columns = X.index)

tfidf_vectorization_mc = mean_center(tfidf_vectorization)
similarity = cosine_similarity_matrix(tfidf_vectorization_mc)
display(similarity.style.format(precision=2).background_gradient())

# Predicting Using kNN Classification

Now that we have the similarity matrix, we can use it to generate **recommendations**. To do so, we first need to look at the **interaction data** between users and documents.

Run the cell below to inspect this data.

In [ ]:
display(impressions.head(20))

The interaction data is organized **per impression event**, where each row corresponds to a single moment at which a user was shown a set of articles.

Here is how to read the columns:

* **ImpressionID**
  A unique identifier for each impression event (used as the index).

* **UserID**
  The identifier of the user who was shown the articles.

* **Time**
  The timestamp at which the impression occurred.

* **History**
  The list of articles the user has previously interacted with (clicked/read) before this impression.
  Sometimes, this column is empty (`NaN`), but in general it provides contextual information about prior user behavior.

* **Impressions**
  A space-separated list of article IDs shown to the user at that moment.
  Each article ID is followed by `-0` or `-1`:

  * `-1` means the user **clicked** the article,
  * `-0` means the article was **shown but not clicked**.

This structure is common for recommender-system datasets: it separates **exposure** (what the system showed) from **interaction** (what the user chose to engage with), which is crucial for evaluating recommendation quality.

## Transform

In this module, we focus only on the **`UserID`** and **`Impressions`** columns. We make the following simplifying assumption:

* If a user **clicked** an article (`Nxxxx-1`), we assume they **liked** it.
* If a user **ignored** an article (`Nxxxx-0`), we assume they **did not like** it.

This turns the interaction data into explicit positive and negative feedback.

### Ignoring session information

Although the data is organized into impression events (sessions), we **ignore session structure** here. Instead, we treat each *(user, article)* pair as an **independent observation**.
This simplifies the problem and allows us to use the same **X / y formulation** as in previous modules. Modeling temporal or session-based effects is important in real systems, but it is beyond the scope of this assignment.

### Target format

We now transform the data into two parts:

* Input data (`X`). The input data should contain one row per *(user, article)* pair, with two columns: `UserID` and `ArticleID`. Each row represents an article that was shown to a user. Example:
  <table border="1" class="dataframe"><thead><tr style="text-align: right;"><th></th><th>UserID</th><th>ArticleID</th></tr></thead><tbody><tr><th>0</th><td>U0019</td><td>N0005</td></tr><tr><th>1</th><td>U0019</td><td>N0006</td></tr><tr><th>2</th><td>U0019</td><td>N0007</td></tr><tr><th>3</th><td>U0019</td><td>N0008</td></tr><tr><th>...</th><td>...</td><td>...</td></tr></tbody></table>
* Target (`y`). The target should indicate whether the user liked the article: `True` if the article was clicked (`-1`) or `False` if it was not clicked (`-0`). Example:
  <table border="1" class="dataframe"><tbody><tr><th>0</th><td>False</td></tr><tr><th>1</th><td>False</td></tr><tr><th>2</th><td>False</td></tr><tr><th>3</th><td>False</td></tr><tr><th>...</th><td>...</td></tr></tbody></table>


With this transformation, we obtain a clean dataset that matches the structure used in earlier kNN modules and can be directly combined with the content-based similarity matrix for prediction.

### Question 6

*4 pts.*

Complete the `transform()` function below. The input is the `impressions` DataFrame. It should produce the `X` DataFrame and `y` Series as described above.

Your function should:

* Convert the `Impressions` column from a string into a list of impression entries (split on whitespace).
* For each row, create one output row per impression entry, producing every *(UserID, ArticleID)* pair the user was shown.
* Split each impression entry on `-` to separate:

  * the article ID (e.g., `N0005`) → goes into `X`, and
  * the click label (`0` or `1`) → goes into `y`.
* Convert the click label into a boolean target:

  * `True` for clicked (`1`)
  * `False` for not clicked (`0`)

The output should be:

* **`X`: a DataFrame** with columns `UserID` and `ArticleID`
* **`y`: a Series** containing booleans (**aligned with `X`**)


In [ ]:
def transform(df: pd.DataFrame) -> (pd.DataFrame, pd.Series):
    # your code here

# Usage:
X, y = transform(impressions)
display(X.head())
display(y.head())

In [ ]:
# test your solution

tests_m2.test_06(transform)

## Train/Test Split

Now that we have the input data `X` and the target `y`, we can create a **train/test split**, just as we did in the previous modules. This allows us to train the model on one part of the data and evaluate it on unseen examples.

You can use the code below to perform this split.

In [ ]:
def train_test_split(X: pd.DataFrame, y: pd.Series, test_size: float = 0.2, random_seed = 42) -> (pd.DataFrame, pd.DataFrame, pd.Series, pd.Series):
    n = len(y)
    ntest = int(test_size * n)
    ntrain = n - ntest

    mask_test = (pd
        .concat([pd.Series([True]*ntest), pd.Series([False]*ntrain)])
        .sample(frac=1, random_state = random_seed))
    mask_test.index = y.index

    X_test = X[mask_test]
    y_test = y[mask_test]
    X_train = X[~mask_test]
    y_train = y[~mask_test]

    return X_train, X_test, y_train, y_test

# apply to the provided data
X_train, X_test, y_train, y_test = train_test_split(X, y)

## Classification

We can use kNN again to make predictions, but there is a fundamental difference from the previous module: we do **not** predict ratings. Instead, our target is **clicked vs. ignored** (as a proxy for liked vs. disliked). This changes the task from **Regression** (predicting continuous values, like ratings), to **Binary classification** (predicting categorical values: `True`/`False`).

Because of this, we use **kNN classification** rather than kNN regression.

### General idea

To predict whether a user will like an article, we look at **similar articles** (based on the TF–IDF similarity matrix) and check whether the user liked those similar articles.

### Step-by-step 

The prediction for one (user, article) pair:

1. **Find the k nearest neighbors of the target article**
   Use the TF–IDF similarity matrix to find the k most similar articles **that the user has already seen**.

2. **Retrieve the user’s labels for those neighbor articles**
   For each neighbor article, look up whether the user **clicked** it (`True`) or **ignored** it (`False`).

3. **Make a majority-vote prediction**
   Instead of computing a (weighted) mean rating, we use a **vote**:

   * if the user clicked the majority of the neighbor articles, predict `True` (will like),
   * otherwise predict `False` (will not like).

This is the classification analogue of what you did earlier with kNN regression: the similarity search is the same, but the aggregation step changes from a weighted average to a vote.

### Question 7

*5 pts.*

Complete the function `knn_recommend()` below. It takes:

* `X_train`, `y_train` — the training user–article pairs and their click labels
* `X_test` — the user–article pairs we want to predict for
* `similarity_df` — the article–article similarity matrix (from TF–IDF)
* `k` — the number of nearest neighbors to use

Your function should run **kNN classification** for every row in `X_test` and return a boolean Pandas `Series` of predicted like/dislike values.

In [ ]:
def knn_recommend(X_train: pd.DataFrame, y_train: pd.Series, X_test: pd.DataFrame, similarity_df: pd.DataFrame, k: int = 5) -> pd.Series:
    # your code here

y_hat = knn_recommend(X_train, y_train, X_test, similarity, 3)
display(y_hat)

In [ ]:
# test your solution
tests_m2.test_07(knn_recommend)

# Evaluation

Now it is time to evaluate the performance of this algorithm.

We start by creating a simple **baseline**: recommend articles based purely on **popularity** (how often they were clicked), regardless of article content.

Run the cell below.

In [ ]:
def popular_baseline(X_train, y_train, X_test):
    train = X_train.copy()
    train["y"] = y_train
    popularity = train.groupby("ArticleID")["y"].sum()
    recommend = popularity > popularity.mean()
    return X_test.join(recommend, on = "ArticleID")["y"]

y_baseline = popular_baseline(X_train, y_train, X_test)

### Question 8

*1 pt.*

Create the confusion matrices below.

In [ ]:
def confusion(y_true: pd.Series, y_pred: pd.Series) -> pd.DataFrame:
    # your code here

confusion_knn = confusion(y_hat, y_test)
display(confusion_knn)

confusion_baseline = confusion(y_baseline, y_test)
display(confusion_baseline)

### Question 9

*1 pt.*

Compute precision for both the predicted values and the baseline.

In [ ]:
def precision(y_true: pd.Series, y_pred: pd.Series) -> float:
    # your code here

precision_knn = precision(y_test, y_hat)
print(precision_knn)
precision_mean = precision(y_test, y_baseline)
print(precision_mean) 

### Question 10

*1 pt.*

Compute recall for both the predicted values and the baseline.

In [ ]:
def recall(y_true: pd.Series, y_pred: pd.Series) -> float:
    # your code here

precision_knn = recall(y_test, y_hat)
print(precision_knn)
precision_mean = recall(y_test, y_baseline)
print(precision_mean) 

# Conclusion

For this small, curated dataset, the content-based kNN approach performs well. Both **precision** and **recall** are high compared to the popularity-based baseline, indicating that the model is able to make meaningful recommendations based on textual similarity.

However, important limitations emerge as we consider scaling this approach to more realistic settings. Article abstracts are relatively short, and TF–IDF relies on **exact word overlap**. As a result, much semantic similarity is missed when different terms are used to describe similar concepts. For example, one article might discuss *“buses”* while another focuses on *“trams”*—topics that are clearly related, but which TF–IDF treats as largely unrelated.

To capture similarity **between terms themselves**, rather than just overlap between words, we need more advanced NLP techniques. These approaches represent words in a semantic space where related terms are close to each other. This is exactly what we will explore in the next notebook.